# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building on ML-04's contract: same tables (`fact_content_daily_performance` +
`dim_content`), same decision date (`2026-03-31`), same two 30-day windows.
This time I engineer 8 features instead of 5 — adding a ratio feature
(CTR), a recency feature (days since last optimized), and proper
categorical + missing-value handling.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/MohinaRustamova/flyrank-ml-internship", "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

%pip -q install duckdb

import duckdb
con = duckdb.connect()

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN: ")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

DECISION_DATE = "2026-03-31"

In [2]:
import pandas as pd
feature_frame = con.sql(f"""
    WITH prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30,
               SUM(gsc_clicks) AS clicks_prior30,
               AVG(gsc_avg_position) AS avg_position_prior30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    current AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_current30,
               SUM(gsc_clicks) AS clicks_current30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT p.client_hash_id, p.content_hash_id,
           p.impressions_prior30, p.clicks_prior30, p.avg_position_prior30,
           c.impressions_current30, c.clicks_current30,
           CASE WHEN c.impressions_current30 < p.impressions_prior30 * 0.8
                THEN 1 ELSE 0 END AS is_declining_label
    FROM prior p
    JOIN current c USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior30 > 0
""").df()

# Engineered ratio feature — guard the divide-by-zero
feature_frame["ctr_prior30"] = (
    feature_frame["clicks_prior30"] / feature_frame["impressions_prior30"]
).replace([float("inf"), -float("inf")], 0).fillna(0)

feats = con.sql(f"""
    SELECT content_hash_id, content_type, word_count,
           DATE_DIFF('day', content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
           last_optimized_date
    FROM {DIMC}
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

feature_frame = feature_frame.merge(feats, on="content_hash_id", how="left")

# Missing-value handling: flag, don't silently fillna(0)
feature_frame["has_word_count"] = feature_frame["word_count"].notna().astype(int)
feature_frame["word_count"] = feature_frame["word_count"].fillna(0)

# Recency feature, leak-safe: an optimization only "counts" if it happened
# ON OR BEFORE the decision date. dim_content reflects CURRENT state, so
# raw last_optimized_date can be in the future relative to 2026-03-31 —
# using that unfiltered would leak information the model couldn't have had.
decision_ts = pd.to_datetime(DECISION_DATE)
last_opt = pd.to_datetime(feature_frame["last_optimized_date"])
known_optimization = last_opt.notna() & (last_opt <= decision_ts)

feature_frame["has_been_optimized"] = known_optimization.astype(int)
feature_frame["days_since_last_optimized"] = (decision_ts - last_opt).dt.days
feature_frame.loc[~known_optimization, "days_since_last_optimized"] = -1  # never optimized as of decision date

# Categorical handling: one-hot, not label-encoded (no false ordering)
feature_frame = pd.get_dummies(feature_frame, columns=["content_type"], prefix="type")

print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(135113, 18)


,client_hash_id,content_hash_id,impressions_prior30,clicks_prior30,avg_position_prior30,impressions_current30,clicks_current30,is_declining_label,ctr_prior30,word_count,content_age_days,last_optimized_date,has_word_count,has_been_optimized,days_since_last_optimized,type_comparison article,type_feedly article,type_keyword article
0,client_e547b89c05043229,content_e67934818ca184a1,320.0,1.0,23.767748,1072.0,2.0,0,0.003125,2905,351.0,2026-05-15,1,0,-1.0,False,False,True
1,client_e547b89c05043229,content_9634c35544bcc47b,411.0,0.0,11.136149,313.0,1.0,1,0.000000,2685,351.0,2026-05-27,1,0,-1.0,False,False,True
2,client_e547b89c05043229,content_4ece07fdea783709,629.0,1.0,10.486653,653.0,0.0,0,0.001590,0,351.0,NaT,0,0,-1.0,False,False,True
3,client_e547b89c05043229,content_4d9b25ca95147676,1421.0,13.0,5.133504,1028.0,18.0,1,0.009148,0,351.0,NaT,0,0,-1.0,False,False,True
4,client_e547b89c05043229,content_2dae25d4660d074a,1209.0,3.0,19.725096,1366.0,1.0,0,0.002481,2758,351.0,2026-05-27,1,0,-1.0,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available when? |
|---|---|---|---|
| `impressions_prior30` | Summed GSC impressions, prior 30d | Rows require `impressions_prior30 > 0` by construction | Before decision date — historical window only |
| `clicks_prior30` | Summed GSC clicks, prior 30d | Same as above | Before decision date |
| `ctr_prior30` | `clicks_prior30 / impressions_prior30` | 0 where impressions were 0 (guarded, not NaN) | Before decision date — derived from prior-window-only inputs |
| `avg_position_prior30` | Average GSC position, prior 30d | Not observed missing in this slice | Before decision date |
| `content_age_days` | Decision date − `content_created_date` | Not observed missing (every page has a creation date) | Always knowable — a fixed fact |
| `days_since_last_optimized` | Decision date − `last_optimized_date`, only counting optimizations that happened on/before the decision date | `-1` = never optimized as of decision date (flagged with `has_been_optimized`, not silently zero) | Before decision date, by construction — future optimizations are excluded |
| `has_been_optimized` | Whether the page had any optimization on/before the decision date | N/A (this IS the missingness flag) | Before decision date |
| `word_count` | Article word count | Missing rows filled to 0, tracked separately via `has_word_count` | Always knowable — static content property |
| `has_word_count` | Whether `word_count` was actually recorded | N/A (this IS the missingness flag) | Always knowable |
| `type_keyword article` / `type_feedly article` / `type_comparison article` | One-hot content type | No missing observed | Always knowable — set at publish time |

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Two leaks hunted here — one deliberate (to prove the test harness works),
one real (found while building the feature vector above).

**Real leak found:** `dim_content.last_optimized_date` reflects the
page's CURRENT state, not its state as of the decision date. Several
rows had optimization dates in May 2026 — two months after my
`2026-03-31` decision date. Using the raw date would have let the
model "know" about a future edit. Fixed in Section 1 by only counting
optimizations that happened on or before the decision date.

**Deliberate leak (test harness check):** add `impressions_current30`
back in as a feature — the exact number the label is computed from —
and confirm the score jumps toward 1.0. If it doesn't, the test itself
is broken.

**Split discipline:** using a random split here would let rows from
the same client appear in both train and test, so the model can
partly memorize per-client behavior instead of learning general
patterns. I split by `client_hash_id` (grouped) instead, and compare
it to a random split to see the size of that gap.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split

honest_cols = [
    "impressions_prior30", "clicks_prior30", "ctr_prior30", "avg_position_prior30",
    "content_age_days", "days_since_last_optimized", "has_been_optimized",
    "word_count", "has_word_count",
    "type_comparison article", "type_feedly article", "type_keyword article",
]

X = feature_frame[honest_cols].fillna(0)
y = feature_frame["is_declining_label"]
groups = feature_frame["client_hash_id"]

base_rate = y.mean()
print(f"Base rate (share declining): {base_rate:.3f}")

# --- Grouped split: honest, no client appears in both train and test ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

grouped_model = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
grouped_score = grouped_model.score(Xte, yte)
print(f"Grouped-split accuracy (honest features): {grouped_score:.3f}  (vs base rate {base_rate:.3f})")

# --- Random split: for comparison, to see the memorization gap ---
Xrtr, Xrte, yrtr, yrte = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
random_model = LogisticRegression(max_iter=1000).fit(Xrtr, yrtr)
random_score = random_model.score(Xrte, yrte)
print(f"Random-split accuracy (honest features):  {random_score:.3f}")
print(f"Gap (random - grouped): {random_score - grouped_score:+.3f}")

# --- Deliberate leak: prove the harness catches it ---
leaky_cols = honest_cols + ["impressions_current30"]
Xl = feature_frame[leaky_cols].fillna(0)
Xltr, Xlte = Xl.iloc[train_idx], Xl.iloc[test_idx]
leaky_model = LogisticRegression(max_iter=1000).fit(Xltr, ytr)
leaky_score = leaky_model.score(Xlte, yte)
print(f"\nLeaky accuracy (+ current30 impressions): {leaky_score:.3f}")
print(f"Jump from leak: {grouped_score:.3f} -> {leaky_score:.3f}")
print("Leak column removed. Keeping the honest, grouped-split number:", round(grouped_score, 3))


Base rate (share declining): 0.240


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Grouped-split accuracy (honest features): 0.712  (vs base rate 0.240)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Random-split accuracy (honest features):  0.759
Gap (random - grouped): +0.047

Leaky accuracy (+ current30 impressions): 1.000
Jump from leak: 0.712 -> 1.000
Leak column removed. Keeping the honest, grouped-split number: 0.712


In [5]:
from sklearn.preprocessing import StandardScaler

# Naive baseline: always guess the majority class
naive_baseline = max(base_rate, 1 - base_rate)
print(f"Naive baseline (always guess majority class): {naive_baseline:.3f}")

scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)

grouped_model = LogisticRegression(max_iter=2000).fit(Xtr_scaled, ytr)
grouped_score = grouped_model.score(Xte_scaled, yte)
print(f"Grouped-split accuracy (honest features, scaled): {grouped_score:.3f}  (vs naive baseline {naive_baseline:.3f})")
print(f"Skill over naive baseline: {grouped_score - naive_baseline:+.3f}")

Naive baseline (always guess majority class): 0.760
Grouped-split accuracy (honest features, scaled): 0.713  (vs naive baseline 0.760)
Skill over naive baseline: -0.046


In [6]:
from sklearn.metrics import roc_auc_score

probs = grouped_model.predict_proba(Xte_scaled)[:, 1]
auc = roc_auc_score(yte, probs)
print(f"ROC-AUC (grouped split): {auc:.3f}  (0.5 = no better than random ranking, 1.0 = perfect ranking)")

# Precision@20 / @50 — the metric your lane actually cares about
results = feature_frame.iloc[test_idx][["content_hash_id", "is_declining_label"]].copy()
results["predicted_prob"] = probs
results_sorted = results.sort_values("predicted_prob", ascending=False)

for k in [20, 50]:
    top_k = results_sorted.head(k)
    precision_at_k = top_k["is_declining_label"].mean()
    print(f"Precision@{k}: {precision_at_k:.3f}  (vs base rate {base_rate:.3f})")

ROC-AUC (grouped split): 0.563  (0.5 = no better than random ranking, 1.0 = perfect ranking)
Precision@20: 0.350  (vs base rate 0.240)
Precision@50: 0.500  (vs base rate 0.240)


Two leaks hunted here — one deliberate (to prove the test harness
works), one real (found while building the feature vector above).

**Real leak found:** `dim_content.last_optimized_date` reflects the
page's CURRENT state, not its state as of the decision date. Several
rows had optimization dates in May 2026 — two months after my
`2026-03-31` decision date. Fixed by only counting optimizations that
happened on or before the decision date.

**Deliberate leak (test harness check):** adding `impressions_current30`
back in — the exact number the label is computed from — pushed accuracy
from 0.706 to 1.000. Confirms the harness catches leakage correctly.

**Split discipline:** a random split scored 0.759 vs. a grouped split
(by `client_hash_id`) at 0.706 — a 0.052 gap, meaning some of the
random-split score was the model partly memorizing per-client behavior
rather than learning general patterns. The grouped number is the
honest one.

**Accuracy is the wrong lens here.** Base rate is 0.240 declining, so
a naive "always guess not-declining" baseline gets 0.760 accuracy —
higher than my scaled model's 0.713. ROC-AUC of 0.563 confirms the
features are weak at ranking the FULL population.

**But the features have real signal at the top of the list** — the
part that actually matters for this lane. Precision@20 = 0.350 and
Precision@50 = 0.500, both well above the 0.240 base rate (roughly
1.5x and 2.1x lift). A reviewer only ever works from the top of a
ranked queue, so this is the metric that counts — and on that metric,
these honest features are doing real work despite looking weak on
accuracy and AUC.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Field | Why excluded |
|---|---|
| `impressions_current30`, `clicks_current30` | Label-derived — this is the number `is_declining_label` is computed from (proven in Section 3) |
| Raw `last_optimized_date` (unfiltered) | Leaks future optimization events past the decision date — replaced with a leak-safe derived version |
| `provider_used`, `model_used` | Data dictionary flags these as not-a-feature |
| `is_published`, `is_deleted` | Used only to filter the slice to live pages — never fed to the model |
| `client_hash_id`, `content_hash_id`, `keyword_hash_id`, `url_hash_id` | Identifiers — used for joining and the grouped split, never as model input |
| `competition_level` | Redundant with the numeric `competition` field; kept out to avoid double-counting the same signal |
| Any FlyRank product flag (health_score, quick-win tags) | Output of an existing rule — using it as a feature would mean learning the old rule, not the world (circular result) |

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.